# Phase 1 — reconstruction-stability segmentation (Kaggle)

Kaggle gives a **predictable 30 h/week quota and 12 h sessions with background execution**,
which suits this job: the full 217-series run takes ~9 h and fits in one session.

## Setup before running
1. **Settings → Accelerator → GPU T4 x2** (or P100).
2. **Settings → Internet → On.** Required to reach TCIA. Kaggle asks for phone verification
   once before it will enable internet.
3. Upload `colab_series_manifest.csv` and `colab_pairs.csv` via **+ Add Input → Upload**,
   or paste them into `/kaggle/working` with the fallback cell below.
4. Then use **Save Version → Save & Run All (Commit)** and close the tab. It runs
   in the background for up to 12 h and `/kaggle/working` is preserved as the version output.

To resume a partial run, attach the previous version's output as an input dataset; step 0
finds `volumes.csv` there automatically.

## 0. Locate inputs and prior state

In [ ]:
import os, glob, shutil
from pathlib import Path
import pandas as pd

WORK = Path("/kaggle/working"); os.chdir(WORK)

# manifests: look in attached inputs, then working dir
def find(name):
    if (WORK/name).exists(): return WORK/name
    hits = glob.glob(f"/kaggle/input/**/{name}", recursive=True)
    return Path(hits[0]) if hits else None

for f in ["colab_series_manifest.csv","colab_pairs.csv"]:
    p = find(f)
    if p and not (WORK/f).exists(): shutil.copy(p, WORK/f); print("found input:", p)
assert (WORK/"colab_series_manifest.csv").exists(), \
    "manifest not found — add it via + Add Input -> Upload"

# prior volumes.csv from an earlier committed version, if attached
if not (WORK/"volumes.csv").exists():
    prior = glob.glob("/kaggle/input/**/volumes.csv", recursive=True)
    if prior: shutil.copy(prior[0], WORK/"volumes.csv"); print("resuming from:", prior[0])

man = pd.read_csv("colab_series_manifest.csv")
if Path("volumes.csv").exists():
    v = pd.read_csv("volumes.csv")
    ok = v[v.error.isna() | v.error.astype(str).str.strip().eq("")]
    print(f"state: {len(v)}/{len(man)} attempted | {len(ok)} ok | {len(man)-len(v)} remaining")
else:
    print(f"state: fresh run | {len(man)} series")

## 1. Environment

## 1a. Preflight — internet and GPU capability

Fails fast with the exact fix. `torch.cuda.is_available()` alone is not enough: it returns
True on a P100, whose sm_60 architecture current torch builds no longer support.


In [ ]:
# --- preflight: both of these must pass before anything is installed ---
import socket, torch, sys

# 1. Internet. Kaggle disables it by default; pip then reports "from versions: none".
try:
    socket.create_connection(("pypi.org", 443), timeout=10).close()
    net = True
except OSError:
    net = False
assert net, ("NO INTERNET. Settings -> Internet -> On (needs phone verification), "
             "then restart the session. TCIA is unreachable without it.")

# 2. GPU must be usable, not merely present. torch.cuda.is_available() returns True
#    on a P100 even though this torch build dropped sm_60 — every kernel launch
#    would then fail at runtime.
assert torch.cuda.is_available(), "No GPU. Settings -> Accelerator -> GPU T4 x2."
name = torch.cuda.get_device_name(0)
cap  = torch.cuda.get_device_capability(0)
supported = torch.cuda.get_arch_list()
print(f"{name} | compute capability {cap[0]}.{cap[1]} | torch {torch.__version__}")
print("torch supports:", supported)
assert cap[0] >= 7, (
    f"{name} is compute capability {cap[0]}.{cap[1]}, which this torch build "
    f"({torch.__version__}) does not support (needs >= 7.0). "
    "Switch Settings -> Accelerator to **GPU T4 x2** (T4 is 7.5) and restart the session.")
print("preflight OK")

In [ ]:
!pip -q install TotalSegmentator==2.18.0 pydicom==3.0.1 nibabel==5.3.2 dicom2nifti 2>&1 | tail -3
from importlib.metadata import version as pkgver
import torch
v = pkgver("TotalSegmentator")          # raises if the install silently failed
print("TotalSegmentator", v, "| torch", torch.__version__,
      "|", torch.cuda.get_device_name(0))
assert v == "2.18.0", f"expected 2.18.0, got {v} — the pin is required by the prespec"

## 2. Provenance

In [ ]:
import json, os, torch
from pathlib import Path
from importlib.metadata import version as pkgver
prov = {"totalsegmentator": pkgver("TotalSegmentator"), "nnunetv2": pkgver("nnunetv2"),
        "torch": torch.__version__, "gpu": torch.cuda.get_device_name(0),
        "platform": "kaggle", "task": "total",
        "structures": ["heart","aorta","pulmonary_vein","superior_vena_cava","inferior_vena_cava"],
        "flags": "-s -sx -sii -ss",
        "fast_mode": False,
        "fast_mode_note": ("--fast resamples to 3.0 mm, which would attenuate the 0.625-5.0 mm "
                           "thickness differences the thickness arm measures. Full-resolution model only.")}
wdir = Path(os.environ["TOTALSEG_WEIGHTS_PATH"]) if "TOTALSEG_WEIGHTS_PATH" in os.environ \
       else Path(os.environ.get("TOTALSEG_HOME_DIR", Path.home()/".totalsegmentator"))/"nnunet/results"
prov["weights_dir"] = str(wdir)
json.dump(prov, open("model_provenance.json","w"), indent=2)
print(json.dumps(prov, indent=1))

## 3. Smoke test — one series

In [ ]:
import subprocess, json, shutil, urllib.request, urllib.parse, zipfile, io, pandas as pd
from pathlib import Path
man = pd.read_csv("colab_series_manifest.csv")
STRUCTS = ["heart","aorta","pulmonary_vein","superior_vena_cava","inferior_vena_cava"]
uid = man.sort_values("n_images").iloc[0].series_uid
dcm, out = Path("smoke_dcm"), Path("smoke_out")
for p in (dcm,out): shutil.rmtree(p, ignore_errors=True); p.mkdir(parents=True, exist_ok=True)
url = "https://services.cancerimagingarchive.net/nbia-api/services/v1/getImage?" + \
      urllib.parse.urlencode({"SeriesInstanceUID": uid})
with urllib.request.urlopen(url, timeout=900) as r:
    zipfile.ZipFile(io.BytesIO(r.read())).extractall(dcm)
cp = subprocess.run(["TotalSegmentator","-i",str(dcm),"-o",str(out),"--task","total",
                     "--roi_subset",*STRUCTS,"-s","-sx","-sii","-ss","--quiet"],
                    capture_output=True, text=True, timeout=3600)
print("exit", cp.returncode, "| files:", [p.name for p in out.iterdir()])
if cp.returncode: print((cp.stderr or cp.stdout)[-800:])
st = json.load(open(out/"statistics.json"))
for s in STRUCTS: print(f"  {s:22s} {st.get(s,{}).get('volume')} mm3")
assert (st.get("heart",{}).get("volume") or 0) > 0, "heart volume 0 — investigate"
for p in (dcm,out): shutil.rmtree(p, ignore_errors=True)
print("\nSMOKE TEST PASSED")

## 4. Full run

Writes `volumes.csv` after every series into `/kaggle/working`, which is preserved as the
version output. Full logs go to `logs/`.

## 4. Full run — set `LIMIT = 2` for the first commit

**Do not commit 5 hours to test plumbing.**

1. Leave `LIMIT = 2` → **Save Version → Save & Run All**. About 5 minutes.
2. Open that version's **Output** tab. If `volumes.csv` is there with 2 rows, output
   capture works — and this tested the real write path, not a proxy for it.
3. Set `LIMIT = None` and commit again for all 217.

The 2 rows are not wasted: the run resumes from `volumes.csv` and skips them.


In [ ]:
# Full run. Two durability guarantees:
#  1. ALL paths are absolute under /kaggle/working — cwd and cell order cannot misplace them.
#  2. Every completed row is ALSO echoed to stdout as CSV. A commit saves its log, so the
#     numbers are recoverable from the log alone even if the output files are lost.
import urllib.request, urllib.parse, json, zipfile, io, shutil, time, subprocess, os, csv, sys
import pandas as pd
from pathlib import Path

OUT = Path("/kaggle/working"); OUT.mkdir(parents=True, exist_ok=True)
MAN = OUT/"colab_series_manifest.csv"
CSV = OUT/"volumes.csv"; LOGS = OUT/"logs"; LOGS.mkdir(exist_ok=True)
assert MAN.exists(), f"manifest missing at {MAN} — run step 0 first"
man = pd.read_csv(MAN)

BASE = "https://services.cancerimagingarchive.net/nbia-api/services/v1"
STRUCTS = ["heart","aorta","pulmonary_vein","superior_vena_cava","inferior_vena_cava"]
FIELDS = (["series_uid","patient","error","resampled","n_slices","runtime_s"] +
          [f"{s}_{k}" for s in STRUCTS for k in ("vol_mm3","nvox","intensity")])

done = set(pd.read_csv(CSV).series_uid) if CSV.exists() else set()

# ---- LIMIT: N = process N series NOT YET DONE (cheap test); None = all remaining ----
LIMIT = 2
if LIMIT:
    todo = man[~man.series_uid.isin(done)].head(LIMIT)
    man = todo.copy()
    print(f'DRY RUN — {len(man)} not-yet-done series')
print(f"writing to {CSV} | skipping {len(done)} already attempted", flush=True)
print("ROWCSV\t" + ",".join(FIELDS), flush=True)      # header for log-based recovery

PRE = ("import dicom2nifti.settings as s; s.disable_validate_orthogonal(); "
       "s.disable_validate_slice_increment(); s.disable_validate_slicecount(); "
       "s.enable_resampling(); s.set_resample_spline_interpolation_order(1); "
       "s.set_resample_padding(-1000)")

def fetch(uid, dest):
    url = f"{BASE}/getImage?" + urllib.parse.urlencode({"SeriesInstanceUID": uid})
    with urllib.request.urlopen(url, timeout=900) as r:
        z = zipfile.ZipFile(io.BytesIO(r.read()))
    dest.mkdir(parents=True, exist_ok=True); z.extractall(dest)
    return sum(1 for n in z.namelist() if n.lower().endswith(".dcm"))

def run_ts(dcm, out, lenient):
    cmd = ["TotalSegmentator","-i",str(dcm),"-o",str(out),"--task","total",
           "--roi_subset",*STRUCTS,"-s","-sx","-sii","-ss","--quiet"]
    if lenient:
        c = f"{PRE}\nimport sys; sys.argv={cmd!r}\nfrom totalsegmentator.bin.TotalSegmentator import main; main()"
        return subprocess.run(["python","-c",c], capture_output=True, text=True, timeout=3600)
    return subprocess.run(cmd, capture_output=True, text=True, timeout=3600)

# any dicom2nifti validation failure is retried leniently, not just NON_CUBICAL
RETRYABLE = ("NON_CUBICAL","SLICE_INCREMENT_INCONSISTENT","MISSING_DICOM_FILES",
             "SLICECOUNT","ConversionValidationError")

t_start = time.time()
for i, row in man.iterrows():
    uid = row.series_uid
    if uid in done: continue
    t0 = time.time(); dcm, seg = OUT/"tmp_dcm", OUT/"tmp_out"
    for p in (dcm,seg): shutil.rmtree(p, ignore_errors=True)
    seg.mkdir(parents=True, exist_ok=True)
    rec = {k: "" for k in FIELDS}
    rec.update({"series_uid": uid, "patient": row.patient, "resampled": False})
    try:
        rec["n_slices"] = fetch(uid, dcm)
        cp = run_ts(dcm, seg, lenient=False)
        blob = (cp.stderr or "") + (cp.stdout or "")
        if cp.returncode != 0 and any(k in blob for k in RETRYABLE):
            cp = run_ts(dcm, seg, lenient=True); rec["resampled"] = True
        (LOGS/f"{uid[-16:]}.log").write_text(f"RC={cp.returncode}\n--OUT--\n{cp.stdout}\n--ERR--\n{cp.stderr}")
        sf = seg/"statistics.json"
        if not sf.exists():
            errs = [l for l in (cp.stderr or "").splitlines() if "Error" in l and "Warning" not in l]
            raise RuntimeError(f"rc={cp.returncode} " + (" | ".join(errs[-2:]) if errs else "no statistics.json"))
        stats = json.load(open(sf))
        for s in STRUCTS:
            d = stats.get(s, {})
            rec[f"{s}_vol_mm3"]=d.get("volume"); rec[f"{s}_nvox"]=d.get("n_voxels")
            rec[f"{s}_intensity"]=d.get("intensity")
    except Exception as e:
        rec["error"] = f"{type(e).__name__}: {str(e)[:200]}".replace(",",";").replace("\n"," ")
        print(f"  !! {uid[-12:]}: {rec['error']}", flush=True)
    rec["runtime_s"] = round(time.time()-t0,1)

    newfile = not CSV.exists()
    with open(CSV,"a",newline="") as f:
        w = csv.DictWriter(f, fieldnames=FIELDS)
        if newfile: w.writeheader()
        w.writerow(rec)
    print("ROWCSV\t" + ",".join(str(rec[k]) for k in FIELDS), flush=True)   # log recovery
    for p in (dcm,seg): shutil.rmtree(p, ignore_errors=True)
    if (i+1) % 20 == 0:
        v = pd.read_csv(CSV)
        nf = int((v.error.notna() & v.error.astype(str).str.strip().ne("")).sum())
        print(f"### {i+1}/{len(man)} | ok {len(v)-nf} | failed {nf} | {(time.time()-t_start)/3600:.1f}h", flush=True)

v = pd.read_csv(CSV)
print(f"\ncomplete: {len(v)}/{len(man)} in {(time.time()-t_start)/3600:.1f} h")
print("files in /kaggle/working:", sorted(p.name for p in OUT.iterdir()))

## 5. Weight hashes and failure rate

In [ ]:
# Weight hashes + failure rate. Self-healing: rebuilds provenance if step 2 was skipped.
import hashlib, json, os, torch, pandas as pd
from pathlib import Path
from importlib.metadata import version as pkgver

def sha256(p, buf=1<<20):
    h = hashlib.sha256()
    with open(p,"rb") as f:
        for b in iter(lambda: f.read(buf), b""): h.update(b)
    return h.hexdigest()

P = Path("model_provenance.json")
prov = json.load(open(P)) if P.exists() else {}
prov.update({"totalsegmentator": pkgver("TotalSegmentator"), "nnunetv2": pkgver("nnunetv2"),
             "torch": torch.__version__, "gpu": torch.cuda.get_device_name(0),
             "platform": "kaggle", "task": "total",
             "structures": ["heart","aorta","pulmonary_vein","superior_vena_cava","inferior_vena_cava"],
             "flags": "-s -sx -sii -ss", "fast_mode": False})
wdir = Path(os.environ["TOTALSEG_WEIGHTS_PATH"]) if "TOTALSEG_WEIGHTS_PATH" in os.environ \
       else Path(os.environ.get("TOTALSEG_HOME_DIR", Path.home()/".totalsegmentator"))/"nnunet/results"
prov["weights_dir"] = str(wdir)

ckpt = sorted(p for ext in ("*.pth","*.pt","*.model") for p in wdir.rglob(ext) if p.is_file()) \
       if wdir.exists() else []
if ckpt:
    prov["weights"] = {str(p.relative_to(wdir)): sha256(p)[:32] for p in ckpt}
    prov["n_weight_files"] = len(ckpt)
    prov["weights_total_MiB"] = round(sum(p.stat().st_size for p in ckpt)/2**20, 1)
    print(f"hashed {len(ckpt)} checkpoint(s), {prov['weights_total_MiB']} MiB")
else:
    print(f"NO WEIGHTS YET under {wdir} — they download on first segmentation. "
          "Re-run this cell after step 3 or 4.")
json.dump(prov, open(P,"w"), indent=2)

if not Path("volumes.csv").exists():
    print("\nvolumes.csv not present — the full run (step 4) has not produced output yet.")
else:
    man = pd.read_csv("colab_series_manifest.csv"); v = pd.read_csv("volumes.csv")
    v["failed"] = v.error.notna() & v.error.astype(str).str.strip().ne("")
    m = v.merge(man[["series_uid","vendor","slice_thickness_mm"]], on="series_uid", how="left")
    print("\nfailure rate: %d/%d (%.1f%%)" % (m.failed.sum(), len(m), 100*m.failed.mean()))
    print(m.groupby("vendor").failed.agg(["sum","count","mean"]).to_string())
    print(m.groupby("slice_thickness_mm").failed.agg(["sum","count","mean"]).to_string())
    if m.failed.any(): print("\n", m.loc[m.failed,"error"].str.slice(0,70).value_counts().to_string())
    m.to_csv("failure_rates.csv", index=False)

## 6. Download (bypasses the Output panel)

`FileLink` renders a clickable link straight from the running session.


In [ ]:
# Download results directly — does not depend on the Output panel.
import shutil, pandas as pd
from pathlib import Path
from IPython.display import FileLink, display
OUT = Path("/kaggle/working")

v = pd.read_csv(OUT/"volumes.csv")
v["failed"] = v.error.notna() & v.error.astype(str).str.strip().ne("")
print(f"rows {len(v)} | ok {int((~v.failed).sum())} | failed {int(v.failed.sum())}")
ok = v[~v.failed]
for s in ["heart","aorta","pulmonary_vein","superior_vena_cava","inferior_vena_cava"]:
    c = f"{s}_vol_mm3"
    if c in ok and ok[c].notna().any():
        ml = ok[c].dropna()/1000.0
        print(f"  {s:20s} median {ml.median():7.1f} mL | range {ml.min():6.1f}-{ml.max():7.1f} | zeros {int((ml==0).sum())}")

for p in ["tmp_dcm","tmp_out"]:                      # leftover scratch
    shutil.rmtree(OUT/p, ignore_errors=True)
z = shutil.make_archive(str(OUT/"phase1_results"), "zip", str(OUT), ".")
print("\nclick to download:")
display(FileLink("phase1_results.zip"))
display(FileLink("volumes.csv"))